Plan:

Evaluate the previously trained nnU-Net 2D and CR-Semi-derived 2D models on the same 61 validation images used for Gemini and BiomedParse.

The evaluation includes:
1. Locating existing predictions and model checkpoints.
2. Generating segmentation predictions if necessary.
3. Extracting connected components and geometric measurements.
4. Preparing figures for manual visual assessment.
5. Comparing segmentation-based and VLM-based annotation methods.

BiomedParse pseudo-labels are not considered expert ground truth

In [6]:
# Mount Google Drive and define project paths.

from google.colab import drive
from pathlib import Path
import pandas as pd

drive.mount("/content/drive")

VLM_DIR = Path(
    "/content/drive/MyDrive/ovarian_ultrasound_vlm"
)

BIOMEDPARSE_DIR = Path(
    "/content/drive/MyDrive/biomedparse_project"
)

NNUNET_RAW_DIR = (
    BIOMEDPARSE_DIR / "nnunet_raw" / "Dataset001_Follicles"
)

MANIFEST_PATH = VLM_DIR / "image_manifest.csv"

print("VLM project:", VLM_DIR.exists())
print("BiomedParse project:", BIOMEDPARSE_DIR.exists())
print("Dataset:", NNUNET_RAW_DIR.exists())
print("Manifest:", MANIFEST_PATH.exists())

Mounted at /content/drive
VLM project: True
BiomedParse project: True
Dataset: True
Manifest: True


Mount Google Drive and define project paths

In [7]:
# Locate existing model checkpoints and segmentation predictions.

checkpoint_extensions = {".pth", ".pt", ".ckpt", ".pkl"}
prediction_extensions = {".nii", ".nii.gz", ".png", ".npz"}

checkpoint_files = []
candidate_prediction_dirs = []

for path in BIOMEDPARSE_DIR.rglob("*"):

    if not path.is_file():
        continue

    name = path.name.lower()
    path_text = str(path).lower()

    # Find model checkpoint files.
    if (
        path.suffix.lower() in checkpoint_extensions
        or "checkpoint" in name
        or "model_best" in name
    ):
        checkpoint_files.append(path)

    # Find possible model prediction directories.
    if any(keyword in path_text for keyword in [
        "prediction",
        "inference",
        "nnunet_results",
        "cr_semi",
        "cr-semi",
    ]):
        if (
            path.suffix.lower() in prediction_extensions
            or name.endswith(".nii.gz")
        ):
            candidate_prediction_dirs.append(path.parent)

print("CHECKPOINT CANDIDATES:")
for path in checkpoint_files[:40]:
    print(path)

print("\nPOSSIBLE PREDICTION DIRECTORIES:")
for directory in sorted(set(candidate_prediction_dirs))[:40]:
    print(directory)

print("\nTotal checkpoint candidates:", len(checkpoint_files))
print("Total candidate directories:", len(set(candidate_prediction_dirs)))

CHECKPOINT CANDIDATES:
/content/drive/MyDrive/biomedparse_project/crsemi_2d_results/checkpoint_best.pth
/content/drive/MyDrive/biomedparse_project/crsemi_2d_results/checkpoint_latest.pth
/content/drive/MyDrive/biomedparse_project/crsemi_2d_fold0_final/checkpoint_epoch_25.pth
/content/drive/MyDrive/biomedparse_project/crsemi_2d_fold0_final/checkpoint_epoch_50.pth
/content/drive/MyDrive/biomedparse_project/crsemi_2d_fold0_final/checkpoint_epoch_75.pth
/content/drive/MyDrive/biomedparse_project/crsemi_2d_fold0_final/checkpoint_epoch_100.pth
/content/drive/MyDrive/biomedparse_project/crsemi_2d_fold0_final/checkpoint_epoch_125.pth
/content/drive/MyDrive/biomedparse_project/crsemi_2d_fold0_final/checkpoint_epoch_150.pth
/content/drive/MyDrive/biomedparse_project/crsemi_2d_fold0_final/checkpoint_epoch_175.pth
/content/drive/MyDrive/biomedparse_project/crsemi_2d_fold0_final/checkpoint_epoch_200.pth
/content/drive/MyDrive/biomedparse_project/crsemi_2d_fold0_final/checkpoint_epoch_225.pth
/conte

Locate existing model checkpoints and segmentation predictions

In [8]:
# Verify the selected model checkpoints and inspect result folders.

NNUNET_FOLD_DIR = (
    BIOMEDPARSE_DIR
    / "nnunet_results"
    / "Dataset001_Follicles"
    / "nnUNetTrainer__nnUNetPlans__2d"
    / "fold_0"
)

CRSEMI_RESULT_DIR = (
    BIOMEDPARSE_DIR
    / "crsemi_2d_fold0_final"
)

NNUNET_CHECKPOINT = NNUNET_FOLD_DIR / "checkpoint_best.pth"
CRSEMI_CHECKPOINT = CRSEMI_RESULT_DIR / "checkpoint_best.pth"

for name, path in [
    ("nnU-Net", NNUNET_CHECKPOINT),
    ("CR-Semi-derived 2D", CRSEMI_CHECKPOINT),
]:
    print(f"\n{name}")
    print("Exists:", path.is_file())
    if path.is_file():
        print("Size (MB):", round(path.stat().st_size / 1024**2, 2))
    print("Path:", path)

print("\nNNUNET FOLD FILES:")
for path in sorted(NNUNET_FOLD_DIR.iterdir()):
    print(path.name)

print("\nCR-SEMI RESULT FILES:")
for path in sorted(CRSEMI_RESULT_DIR.iterdir()):
    print(path.name)


nnU-Net
Exists: True
Size (MB): 353.63
Path: /content/drive/MyDrive/biomedparse_project/nnunet_results/Dataset001_Follicles/nnUNetTrainer__nnUNetPlans__2d/fold_0/checkpoint_best.pth

CR-Semi-derived 2D
Exists: True
Size (MB): 89.74
Path: /content/drive/MyDrive/biomedparse_project/crsemi_2d_fold0_final/checkpoint_best.pth

NNUNET FOLD FILES:
checkpoint_best.pth
checkpoint_best100.pth
checkpoint_latest.pth
checkpoint_latest100.pth
debug.json
progress.png
training_log_2026_10_10_12_55_48.txt
training_log_2026_10_10_15_33_55.txt
training_log_2026_10_11_07_39_20.txt
training_log_2026_10_7_20_10_18.txt
training_log_2026_10_7_20_25_29.txt

CR-SEMI RESULT FILES:
checkpoint_best.pth
checkpoint_epoch_100.pth
checkpoint_epoch_125.pth
checkpoint_epoch_150.pth
checkpoint_epoch_175.pth
checkpoint_epoch_200.pth
checkpoint_epoch_225.pth
checkpoint_epoch_25.pth
checkpoint_epoch_250.pth
checkpoint_epoch_275.pth
checkpoint_epoch_300.pth
checkpoint_epoch_50.pth
checkpoint_epoch_75.pth
checkpoint_latest.p

Verify the selected model checkpoints and inspect result folders

In [9]:
# Inspect model checkpoints using restricted PyTorch loading.

import torch
import numpy as np
from numpy._core.multiarray import scalar
from torch.serialization import safe_globals

numpy_types = [
    scalar,
    np.dtype,
    np.core.multiarray._reconstruct,
    np.ndarray,
    np.dtypes.Float32DType,
    np.dtypes.Float64DType,
]

def inspect_checkpoint(name, checkpoint_path):
    print(f"\n{'=' * 55}")
    print(name)
    print(f"{'=' * 55}")

    with safe_globals(numpy_types):
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=True
        )

    print("Checkpoint type:", type(checkpoint).__name__)

    if isinstance(checkpoint, dict):
        print("\nTop-level keys:")
        for key in checkpoint:
            print("-", key)

        for key in [
            "epoch",
            "current_epoch",
            "best_dice",
            "best_val_dice",
            "best_metric",
            "trainer_name",
        ]:
            if key in checkpoint:
                print(f"{key}: {checkpoint[key]}")

    del checkpoint


inspect_checkpoint(
    "nnU-Net 2D — best",
    NNUNET_CHECKPOINT
)

inspect_checkpoint(
    "nnU-Net 2D — best100",
    NNUNET_FOLD_DIR / "checkpoint_best100.pth"
)

inspect_checkpoint(
    "CR-Semi-derived 2D — best",
    CRSEMI_CHECKPOINT
)

/tmp/ipykernel_1382/4043717459.py:11: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  np.core.multiarray._reconstruct,



nnU-Net 2D — best
Checkpoint type: dict

Top-level keys:
- network_weights
- optimizer_state
- grad_scaler_state
- logging
- _best_ema
- current_epoch
- init_args
- trainer_name
- inference_allowed_mirroring_axes
current_epoch: 160
trainer_name: nnUNetTrainer

nnU-Net 2D — best100
Checkpoint type: dict

Top-level keys:
- network_weights
- optimizer_state
- grad_scaler_state
- logging
- _best_ema
- current_epoch
- init_args
- trainer_name
- inference_allowed_mirroring_axes
current_epoch: 92
trainer_name: nnUNetTrainer

CR-Semi-derived 2D — best
Checkpoint type: dict

Top-level keys:
- epoch
- model_state_dict
- optimizer_state_dict
- train_loss
- train_dice
- val_loss
- val_dice
- best_dice
epoch: 231
best_dice: 0.4311909124922563


Inspect model checkpoints using restricted PyTorch loading

In [10]:
# Inspect nnU-Net validation metrics stored in checkpoints.

from torch.serialization import safe_globals

for name, path in [
    ("best", NNUNET_CHECKPOINT),
    ("best100", NNUNET_FOLD_DIR / "checkpoint_best100.pth"),
]:
    with safe_globals(numpy_types):
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=True
        )

    print(f"\n{name}")
    print("Epoch:", checkpoint["current_epoch"])
    print("Best EMA:", checkpoint.get("_best_ema"))

    logging = checkpoint.get("logging", {})

    print("Logging keys:", list(logging.keys()))

    for key, value in logging.items():
        if "dice" in key.lower() or "ema" in key.lower():
            if isinstance(value, list):
                print(f"{key}: last 5 values = {value[-5:]}")
            else:
                print(f"{key}: {value}")

    del checkpoint


best
Epoch: 160
Best EMA: 0.46347475
Logging keys: ['mean_fg_dice', 'ema_fg_dice', 'dice_per_class_or_region', 'train_losses', 'val_losses', 'lrs', 'epoch_start_timestamps', 'epoch_end_timestamps']
mean_fg_dice: last 5 values = [np.float32(0.49690965), np.float32(0.4290448), np.float32(0.49948758), np.float32(0.46802047), np.float32(0.47133994)]
ema_fg_dice: last 5 values = [np.float32(0.461032), np.float32(0.4578333), np.float32(0.4619987), np.float32(0.46260086), np.float32(0.46347475)]
dice_per_class_or_region: last 5 values = [[np.float32(0.49690965)], [np.float32(0.4290448)], [np.float32(0.49948758)], [np.float32(0.46802047)], [np.float32(0.47133994)]]

best100
Epoch: 92
Best EMA: 0.4592603
Logging keys: ['mean_fg_dice', 'ema_fg_dice', 'dice_per_class_or_region', 'train_losses', 'val_losses', 'lrs', 'epoch_start_timestamps', 'epoch_end_timestamps']
mean_fg_dice: last 5 values = [np.float32(0.4055398), np.float32(0.524481), np.float32(0.4724933), np.float32(0.47213012), np.float32

Inspect nnU-Net validation metrics stored in checkpoints

In [11]:
# Check nnU-Net installation, GPU availability, and model configuration.

import torch
import importlib.util
from pathlib import Path

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print(
    "nnU-Net installed:",
    importlib.util.find_spec("nnunetv2") is not None
)

NNUNET_MODEL_DIR = NNUNET_FOLD_DIR.parent

for filename in [
    "dataset.json",
    "plans.json",
]:
    path = NNUNET_MODEL_DIR / filename
    print(filename, "exists:", path.exists())

print(
    "Best checkpoint exists:",
    NNUNET_CHECKPOINT.exists()
)

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
nnU-Net installed: True
dataset.json exists: True
plans.json exists: True
Best checkpoint exists: True


Check nnU-Net installation, GPU availability, and model configuration

In [12]:
# Prepare the same 61 validation images for nnU-Net inference.

import shutil
from pathlib import Path

NNUNET_INPUT_DIR = Path("/content/nnunet_validation_input")
NNUNET_OUTPUT_DIR = Path("/content/nnunet_validation_predictions")

NNUNET_DRIVE_OUTPUT = (
    VLM_DIR / "predictions" / "nnunet_2d_validation"
)

NNUNET_INPUT_DIR.mkdir(parents=True, exist_ok=True)
NNUNET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NNUNET_DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_PATH)

validation = manifest.loc[
    manifest["split"].astype(str).str.strip().str.lower()
    .isin(["val", "validation"])
].copy()

assert len(validation) == 61
assert validation["case_id"].is_unique

# Copy validation images to the local Colab runtime.
for row in validation.itertuples(index=False):
    source = Path(row.image_path)
    target = NNUNET_INPUT_DIR / f"{row.case_id}_0000.png"

    assert source.exists(), f"Missing image: {source}"

    if not target.exists():
        shutil.copy2(source, target)

input_files = list(NNUNET_INPUT_DIR.glob("*_0000.png"))

assert len(input_files) == 61, (
    f"Expected 61 input images, found {len(input_files)}"
)

print("Prepared images:", len(input_files))
print("Input directory:", NNUNET_INPUT_DIR)

Prepared images: 61
Input directory: /content/nnunet_validation_input


Prepare the 61 validation images for inference using the previously trained nnU-Net 2D model

In [13]:
# Install nnU-Net v2 in the current Colab Python environment (for inference)

import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "nnunetv2"],
    check=True
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'nnunetv2'], returncode=0)

Install nnU-Net v2 in the current Colab Python environment.

In [14]:
# Verify nnU-Net installation and CUDA support.

import sys
import shutil
import torch
import nnunetv2

print("Python:", sys.executable)
print("nnU-Net module:", nnunetv2.__file__)
print("Prediction command:", shutil.which("nnUNetv2_predict"))
print("CUDA available:", torch.cuda.is_available())

Python: /usr/bin/python3
nnU-Net module: /usr/local/lib/python3.13/dist-packages/nnunetv2/__init__.py
Prediction command: /usr/local/bin/nnUNetv2_predict
CUDA available: True


In [15]:
# Run inference with the previously trained nnU-Net 2D checkpoint.

import os
import subprocess
import sys

env = os.environ.copy()

env["nnUNet_raw"] = str(BIOMEDPARSE_DIR / "nnunet_raw")
env["nnUNet_preprocessed"] = str(BIOMEDPARSE_DIR / "nnunet_preprocessed")
env["nnUNet_results"] = str(BIOMEDPARSE_DIR / "nnunet_results")

# Prevent accidental overwriting of existing predictions.
existing_predictions = list(NNUNET_OUTPUT_DIR.glob("*.png"))

assert not existing_predictions, (
    "Prediction files already exist. Inspect them before rerunning inference."
)

command = [
    sys.executable,
    "-c",
    "from nnunetv2.inference.predict_from_raw_data "
    "import predict_entry_point; predict_entry_point()",
    "-i", str(NNUNET_INPUT_DIR),
    "-o", str(NNUNET_OUTPUT_DIR),
    "-d", "Dataset001_Follicles",
    "-c", "2d",
    "-f", "0",
    "-chk", "checkpoint_best.pth",
    "-device", "cuda",
    "-npp", "1",
    "-nps", "1"
]

print("Running nnU-Net inference...")

# Run inference and display the actual nnU-Net error.

result = subprocess.run(
    command,
    env=env,
    capture_output=True,
    text=True,
    check=False
)

print("Return code:", result.returncode)

print("\nSTDOUT:")
print(result.stdout[-5000:])

print("\nSTDERR:")
print(result.stderr[-8000:])

if result.returncode == 0:
    print("\nInference completed successfully.")
else:
    print("\nInference failed. See the error details above.")

print("Inference completed.")

Running nnU-Net inference...
Return code: 0

STDOUT:
minant_0121:
perform_everything_on_device: True
sending off prediction to background worker for resampling and export
done with dominant_0121

Predicting dominant_0133:
perform_everything_on_device: True
sending off prediction to background worker for resampling and export
done with dominant_0133

Predicting dominant_0141:
perform_everything_on_device: True
sending off prediction to background worker for resampling and export
done with dominant_0141

Predicting normal_0002:
perform_everything_on_device: True
sending off prediction to background worker for resampling and export
done with normal_0002

Predicting normal_0005:
perform_everything_on_device: True
sending off prediction to background worker for resampling and export
done with normal_0005

Predicting normal_0007:
perform_everything_on_device: True
sending off prediction to background worker for resampling and export
done with normal_0007

Predicting normal_0009:
perform_ever

Run inference with the previously trained nnU-Net 2D checkpoint

In [16]:
# Validate and save nnU-Net predictions to Google Drive.

from pathlib import Path
from PIL import Image
import numpy as np
import shutil

NNUNET_OUTPUT_DIR = Path("/content/nnunet_validation_predictions")

NNUNET_DRIVE_OUTPUT = (
    VLM_DIR / "predictions" / "nnunet_2d_validation"
)

predicted_files = sorted(NNUNET_OUTPUT_DIR.glob("*.png"))

print("Predicted masks:", len(predicted_files))

# Check that all 61 validation cases are present.
expected_names = {
    f"{case_id}.png"
    for case_id in validation["case_id"]
}

actual_names = {path.name for path in predicted_files}

assert actual_names == expected_names, (
    f"Missing: {expected_names - actual_names}; "
    f"Unexpected: {actual_names - expected_names}"
)

# Check mask dimensions and label values.
for row in validation.itertuples(index=False):
    prediction_path = NNUNET_OUTPUT_DIR / f"{row.case_id}.png"

    mask = np.array(Image.open(prediction_path))
    image = np.array(Image.open(row.image_path))

    assert mask.shape == image.shape[:2], row.case_id
    assert set(np.unique(mask)).issubset({0, 1}), row.case_id

print("All 61 masks passed validation.")

# Save predictions without overwriting existing files.
NNUNET_DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

for path in predicted_files:
    destination = NNUNET_DRIVE_OUTPUT / path.name

    if destination.exists():
        print("Already saved:", path.name)
    else:
        shutil.copy2(path, destination)

print("Google Drive:", NNUNET_DRIVE_OUTPUT)
print(
    "Saved masks:",
    len(list(NNUNET_DRIVE_OUTPUT.glob("*.png")))
)

Predicted masks: 61
All 61 masks passed validation.
Google Drive: /content/drive/MyDrive/ovarian_ultrasound_vlm/predictions/nnunet_2d_validation
Saved masks: 61


Validate and save nnU-Net predictions to Google Drive